# ODDS Real-Data Benchmark (Preview)

Load a classic ODDS dataset, standardize, and compare a few PyOD detectors.
For the full suite across all bundled datasets, prefer:

```bash
python benchmark_odds.py
```

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
from pyod.models.hbos import HBOS
from pyod.models.iforest import IForest
from pyod.models.knn import KNN
from pyod.models.lof import LOF
from pyod.models.ocsvm import OCSVM
from pyod.utils.utility import standardizer
from sklearn.model_selection import train_test_split

from utils.data_loading import describe_dataset, load_odds_mat
from utils.metrics import evaluate_detector, summarize_results

In [ ]:
dataset = 'cardio'  # try: ionosphere, arrhythmia, pima
X, y, meta = load_odds_mat(dataset)
describe_dataset(dataset)

contamination = float(meta['contamination'])
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.4, random_state=42, stratify=y
)
X_train, X_test = standardizer(X_train, X_test)

In [ ]:
classifiers = {
    'IForest': IForest(contamination=contamination, random_state=42),
    'HBOS': HBOS(contamination=contamination),
    'KNN': KNN(contamination=contamination),
    'LOF': LOF(n_neighbors=20, contamination=contamination),
    'OCSVM': OCSVM(contamination=contamination),
}

results = {}
for name, clf in classifiers.items():
    clf.fit(X_train)
    results[name] = evaluate_detector(
        y_test, clf.predict(X_test), clf.decision_function(X_test)
    )

summarize_results(results)
pd.DataFrame(results).T.sort_values('roc_auc', ascending=False)

## Cached full-benchmark table

Precomputed means across `cardio` / `ionosphere` / `arrhythmia` / `pima`
are stored in `../results/odds_benchmark_summary.csv`.

In [ ]:
summary = pd.read_csv('../results/odds_benchmark_summary.csv', index_col=0)
summary.sort_values('roc_auc', ascending=False)